# 🧠 AI Logic Engine: Main Inference Console

Logic Graph ကို အခြေခံ၍ Concept နှစ်ခုအကြား ဆက်သွယ်မှု (Inference) များကို ထုတ်ပေးရန်။

In [ ]:
# Step 1: Initialize
!pip install -q firebase-admin

from google.colab import files
import firebase_admin
from firebase_admin import credentials, firestore
import re

DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2'

if not firebase_admin._apps:
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

db = firestore.client(database_id=DATABASE_ID)
print("✅ Memory Core Ready.")

In [ ]:
# Step 2: Core Logic engine
class KnowledgeInferenceEngine:
    def __init__(self, db):
        self.db = db
        self.cache = {}

    def clean_id(self, text):
        if not text: return ""
        text = text.lower().strip()
        return re.sub(r'[^a-z0-9]+', '_', text).strip('_')

    def solve(self, start, target, depth=10):
        sid, tid = self.clean_id(start), self.clean_id(target)
        queue = [(sid, [])]
        visited = {sid}
        while queue:
            curr, path = queue.pop(0)
            if curr == tid: return path
            if len(path) >= depth: continue
            
            doc = self.db.collection('nodes').document(curr).get()
            if not doc.exists: continue
            data = doc.to_dict()
            
            for g in data.get('groups', []):
                if g not in visited:
                    visited.add(g)
                    queue.append((g, path + [f"{curr} is type of {g}"]))
            for r in data.get('relations', []):
                target_id = r.get('targetId')
                if target_id and target_id not in visited:
                    visited.add(target_id)
                    queue.append((target_id, path + [r.get('sentence', f"{curr} {r['verb']} {target_id}")]))
        return None

engine = KnowledgeInferenceEngine(db)
print("✅ Logic Engine Stabilized.")

In [ ]:
# Step 3: Run Proof
A = "Mitochondria"
B = "Cellular Energy"

proof = engine.solve(A, B)
if proof:
    for step in proof: print(f"-> {step}")
else:
    print("❌ Path not found.")